# wScReNI inference + precision/recall on the SeaAD paired multiome

Mirrors `run_infer_wscreni.ipynb` (retinal demo), adapted for the SeaAD-paired Phase 2/3 outputs produced by the team's pipeline:

1. Load the processed SeaAD paired inputs (RNA, peaks, KNN, triplets, gene labels).
2. Build the `GenePeakOverlapLabs` object.
3. Run `infer_wscreni_networks` → per-cell weighted gene×gene regulatory networks.
4. Evaluate against the **human** ChIP-Atlas ground truth (`chip_seq_5kb_TF_target.df.txt`): precision/recall at several top-K cutoffs.

**Scale note:** SeaAD-paired `_sub42` has 1,200 cells × 500 HVGs (3× the retinal demo). wScReNI is per-cell so wall time scales linearly. With `n_jobs=32` this is several hours on a typical SLURM node — submit via the matching SLURM wrapper rather than running interactively.

**Differences from `run_infer_wscreni.ipynb`:**
- File-naming convention: `seaad_paired_*_sub42.h5ad` / `seaad_paired_sub42_*.csv` (the team's Phase 2/3 uses an explicit seed suffix).
- KNN is stored inside `_sub42.h5ad`'s `.uns['knn_indices']`, not as a separate `.npy` file.
- ChIP-Atlas is the human file, not the mouse one.
- Output directory is namespaced to `output/networks_seaad_paired/` so it doesn't collide with the retinal run.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import anndata as ad

# Make the local screni package importable when running from src/screni/data/.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

from screni.data import (
    GenePeakOverlapLabs,
    infer_wscreni_networks,
    calculate_precision_recall,
    load_chip_atlas,
)

In [ ]:
# ---- Configuration ----
DATASET   = "seaad_paired"                 # 'seaad_paired' is the only branch wired up here
SUB_TAG   = "sub42"                          # Phase 2 seed embedded in file names (sub42 = seed 42)
DATA_DIR  = os.path.join(REPO_ROOT, "data", "processed", "seaad")
REF_PATH  = os.path.join(REPO_ROOT, "data", "chip_seq_5kb_TF_target.df.txt")   # HUMAN ChIP-Atlas
OUT_DIR   = os.path.join(REPO_ROOT, "src", "screni", "data", "output")
NET_DIR   = os.path.join(OUT_DIR, f"networks_{DATASET}")                       # per-dataset output dir
os.makedirs(OUT_DIR, exist_ok=True)

# Phase 2 outputs (matches data/processed/seaad/ on the cluster)
RNA_PATH       = os.path.join(DATA_DIR, f"{DATASET}_rna_{SUB_TAG}.h5ad")
ATAC_PATH      = os.path.join(DATA_DIR, f"{DATASET}_atac_{SUB_TAG}.h5ad")
# Phase 3 outputs (note the slightly different prefix layout)
TRIPLETS_PATH  = os.path.join(DATA_DIR, f"{DATASET}_{SUB_TAG}_triplets.csv")
LABELS_PATH    = os.path.join(DATA_DIR, f"{DATASET}_{SUB_TAG}_gene_labels.csv")
PEAK_MAT_PATH  = os.path.join(DATA_DIR, f"{DATASET}_{SUB_TAG}_peak_overlap_matrix.npz")

# Compute knobs
N_JOBS              = int(os.environ.get("WSCRENI_N_JOBS", "32"))
MAX_CELLS_PER_BATCH = 10

for label, path in [
    ("RNA", RNA_PATH), ("ATAC", ATAC_PATH),
    ("triplets", TRIPLETS_PATH), ("gene_labels", LABELS_PATH),
    ("peak_matrix", PEAK_MAT_PATH), ("ChIP-Atlas", REF_PATH),
]:
    exists = "✓" if os.path.exists(path) else "✗"
    print(f"  {exists} {label}: {path}")
print(f"\noutputs -> {NET_DIR}")
print(f"n_jobs   = {N_JOBS}")

In [ ]:
# Load processed inputs.
rna       = ad.read_h5ad(RNA_PATH)                    # (1200 cells, 500 genes)
triplets  = pd.read_csv(TRIPLETS_PATH)                # TF | peak | target_gene | spearman_r
labels_df = pd.read_csv(LABELS_PATH)                  # gene | type | associated_peaks | associated_TFs
peak_matrix = np.load(PEAK_MAT_PATH)["peak_matrix"]    # (cells, n_unique_peaks)
peak_names  = triplets["peak"].unique().tolist()

# KNN is in the subsampled RNA's .uns (SeaAD pipeline stores it there, not as a separate .npy).
if "knn_indices" not in rna.uns:
    raise KeyError(
        f"{RNA_PATH} has no uns['knn_indices']. "
        f"Re-run Phase 2 (feature_selection.py) with KNN export enabled."
    )
knn = np.asarray(rna.uns["knn_indices"])               # (cells, K) 0-based

print(f"RNA        : {rna.shape}  (cells x genes)")
print(f"peak_matrix: {peak_matrix.shape}  (cells x peaks)")
print(f"KNN        : {knn.shape}  (cells x neighbours)")
print(f"triplets   : {triplets.shape}; gene labels: {labels_df.shape}")
print(f"unique peaks across triplets: {len(peak_names)}")

In [ ]:
# Build the (gene, peak, TF) label table the inference function consumes.
# Identical to the retinal notebook — only the inputs are SeaAD.
rows = []
for _, row in labels_df.iterrows():
    if pd.isna(row["associated_peaks"]):
        continue
    peaks_for_gene = str(row["associated_peaks"]).split(",")
    tf_str = (
        ";".join(str(row["associated_TFs"]).split(","))
        if pd.notna(row["associated_TFs"]) else ""
    )
    for peak in peaks_for_gene:
        rows.append({"gene": row["gene"], "peak": peak.strip(), "TF": tf_str, "label": row["type"]})

labs_df = pd.DataFrame(rows)

labs = GenePeakOverlapLabs(
    genes  = labs_df["gene"].tolist(),
    peaks  = labs_df["peak"].tolist(),
    tfs    = labs_df["TF"].tolist(),     # semicolon-separated
    labels = labs_df["label"].tolist(),  # 'TF' or 'target'
)
print(f"labs entries: {len(labs.genes)}")
print(f"TF genes    : {(labels_df['type'] == 'TF').sum()}, target genes: {(labels_df['type'] == 'target').sum()}")

In [ ]:
# Run wScReNI inference.
# Output: ScReniNetworks dict keyed by cell barcode -> (500, 500) weight matrix.
# Per-cell files are also written to NET_DIR/wScReNI/<idx>.<cell>.network.txt
# (combine_wscreni_networks can reload them later).
networks = infer_wscreni_networks(
    expr                  = rna,                # AnnData, cells x genes
    peak_mat              = peak_matrix,        # ndarray, cells x peaks
    labs                  = labs,
    nearest_neighbors_idx = knn,
    network_path          = NET_DIR,
    data_name             = DATASET,
    cell_index            = None,               # None = all cells
    n_jobs                = N_JOBS,
    max_cells_per_batch   = MAX_CELLS_PER_BATCH,
    peak_names            = peak_names,
    cell_names            = rna.obs_names.tolist(),
)

print(f"Inferred {len(networks)} networks. First matrix shape: {next(iter(networks.values())).shape}")

In [ ]:
# Checkpoint: attach gene order to the AnnData + save for downstream analysis.
first_cell = next(iter(networks))
print(f"first cell : {first_cell}")
print(f"first matrix: {networks[first_cell].shape}, "
      f"non-zero edges: {int(np.count_nonzero(networks[first_cell]))}")

rna.uns["wScReNI_gene_order"] = list(networks.gene_names)
rna.write_h5ad(os.path.join(OUT_DIR, f"{DATASET}_with_networks.h5ad"))

## Evaluate against the human ChIP-Atlas ground truth

`chip_seq_5kb_TF_target.df.txt` is the human ChIP-Atlas TF→target table (±5 kb of TSS). Same evaluation protocol as the retinal notebook:

1. Rank the (gene, gene) edges of each per-cell weight matrix by weight.
2. Take the top-K edges.
3. Check how many appear in the ChIP-Atlas set.
4. precision = TP / K; recall = TP / (TPs across the full 500×500 edge list).

Numbers will be lower than retinal because the gene set is different (human HVGs vs mouse retinal HVGs), and human ChIP-Atlas is much denser, so the recall ceiling shifts. The shape of the precision/recall vs K curve should still match the paper's findings.

In [ ]:
tf_pairs   = load_chip_atlas(REF_PATH)
gene_list  = list(networks.gene_names)
gene_set   = set(gene_list)

# Reachable ground truth: pairs where both TF and target are in our 500-gene universe.
reachable = {p for p in tf_pairs
             if len(p.split("_", 1)) == 2
             and p.split("_", 1)[0] in gene_set
             and p.split("_", 1)[1] in gene_set}
print(f"ChIP-Atlas total pairs        : {len(tf_pairs):,}")
print(f"Reachable in this 500-gene set: {len(reachable):,}  (recall ceiling)")

In [ ]:
# Precision/recall per cell at several K values.
# 1200 cells x 9 K values = ~11k calls; each call is O(n^2) on a 500x500 matrix.
TOP_KS = [100, 250, 500, 1000, 2000, 5000, 10000, 20000, 35000]
records = []
for cell, mat in networks.items():
    for k in TOP_KS:
        p, r = calculate_precision_recall(
            scnetwork_weights = mat,
            tf_target_pair    = tf_pairs,
            top_number        = k,
            gene_names        = gene_list,
        )
        records.append({"cell": cell, "K": k, "precision": p, "recall": r})

pr_df = pd.DataFrame(records)
pr_df.to_csv(os.path.join(OUT_DIR, f"{DATASET}_wscreni_precision_recall.csv"), index=False)
print(f"Computed {len(pr_df)} (cell, K) results -> {DATASET}_wscreni_precision_recall.csv")
pr_df.head()

In [ ]:
# Mean / std across cells per K — the supervisor-meeting summary table.
summary = (
    pr_df.groupby("K")[["precision", "recall"]]
         .agg(["mean", "std"])
         .round(4)
)
summary.to_csv(os.path.join(OUT_DIR, f"{DATASET}_wscreni_pr_summary.csv"))
print(summary)

In [ ]:
# PR curve view (mean across cells).
import matplotlib.pyplot as plt
mean_pr = pr_df.groupby("K")[["precision", "recall"]].mean().reset_index()

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(mean_pr["recall"], mean_pr["precision"], marker="o", label="wScReNI (SeaAD paired)")

for _, row in mean_pr.iterrows():
    ax.annotate(f"K={int(row['K'])}", (row["recall"], row["precision"]),
                textcoords="offset points", xytext=(5, 5), fontsize=8)

random_p = len(reachable) / (len(gene_list) ** 2)
ax.axhline(random_p, ls="--", color="gray", alpha=0.6,
           label=f"Random baseline ({random_p:.4f})")

ax.set_xlabel("Recall (mean across cells)")
ax.set_ylabel("Precision (mean across cells)")
ax.set_title(f"wScReNI vs human ChIP-Atlas — {DATASET}")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc="best")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, f"{DATASET}_wscreni_pr_curve.png"), dpi=120)
plt.show()

## Next step

Once this notebook completes:
- Per-cell networks live in `output/networks_seaad_paired/wScReNI/`.
- The AnnData checkpoint `seaad_paired_with_networks.h5ad` stores the gene order.
- `run_differential_grn.ipynb` consumes these to produce the ranked TF→target table per cell type (SQ1).